[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PacktPublishing/Data-Strategy-for-LLMs/blob/main/chapter_09/Jupyter_Notebooks/Chapter_9_Notebook.ipynb)

**Click the badge above to run this notebook in Google Colab (no local setup needed).**


# Chapter 9: Task-Specific Evaluation Data

This notebook evaluates the outputs from Chapter 6 (Synthetic Data Generation) and Chapter 7 (Post Training).
Instead of talking about evaluation in the abstract, we load the actual datasets those chapters produced and measure whether they are any good.

**What we evaluate:**
1. Chapter 6 synthetic QA pairs - are they faithful to the source? Are they diverse enough?
2. Chapter 6 preference pairs - can an LLM judge tell which response is better?
3. Chapter 7 SFT data - does the fine-tuned model actually improve over the base?

**Run the Shared Setup cell first.**

## Setup

**IMPORTANT: This chapter uses the book-wide shared environment. Follow the README.md in the repository root.**

**Before running:**

1. Run the book-wide setup once from the repository root:
   - macOS/Linux: `bash setup/setup_mac.sh`
   - Windows (PowerShell): `powershell -ExecutionPolicy Bypass -File setup/setup_windows.ps1`

   This creates the `data_strategy_env` environment, registers the **"Python (Data Strategy Book)"** Jupyter kernel, and configures your API key.

2. Select the **"Python (Data Strategy Book)"** kernel (top-right). If it is not listed: Command Palette -> "Developer: Reload Window".

3. This chapter evaluates datasets produced by **Chapter 6** and **Chapter 7**. Those datasets are already committed in the repo; if you regenerate them, run the Ch 6 and Ch 7 notebooks first.

The next cell installs any missing packages **into the running kernel** and loads your API key (it prompts you if no `.env` key is found, e.g., on Colab). Then run all cells.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
# === Chapter 9 Setup: kernel-aware install + API key ===
# Works on local ("Python (Data Strategy Book)" kernel), Google Colab, and fresh environments.
import sys, subprocess
from pathlib import Path

def _install(pkg):
    """Install into the RUNNING kernel's Python (not the shell pip), with fallbacks."""
    for cmd in (
        [sys.executable, "-m", "pip", "install", pkg, "--quiet"],
        [sys.executable, "-m", "pip", "install", pkg, "--user", "--quiet"],
        [sys.executable, "-m", "pip", "install", pkg, "--break-system-packages", "--quiet"],
    ):
        try:
            subprocess.run(cmd, check=True, capture_output=True, text=True)
            return True
        except subprocess.CalledProcessError:
            continue
    return False

for _pkg in ("openai", "pandas", "numpy", "python-dotenv"):
    if not _install(_pkg):
        print(f"WARNING: could not install {_pkg} (restart the kernel and re-run this cell)")

import os, json
import pandas as pd
import numpy as np
from openai import OpenAI
from collections import Counter

# --- Load the OpenAI API key the same way as the rest of the book (utils/config.py) ---
repo_root = Path.cwd()
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / "utils" / "config.py").exists():
        repo_root = _p
        break
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    from utils.config import get_openai_api_key
    api_key = get_openai_api_key()          # searches up for .env, raises a helpful error if missing
except Exception:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        import getpass
        api_key = getpass.getpass("Enter your OpenAI API key: ")   # Colab / no-.env fallback
os.environ["OPENAI_API_KEY"] = api_key

client = OpenAI(api_key=api_key)
JUDGE_MODEL = "gpt-4o-mini"

# Discover the latest available model instead of hardcoding a stale one
def get_best_available_model():
    """Find the best available OpenAI model for evaluation tasks."""
    preferred = ["gpt-4o", "gpt-4o-mini", "gpt-4-turbo", "gpt-4", "gpt-3.5-turbo"]
    try:
        available = {m.id for m in client.models.list()}
        for model in preferred:
            if model in available:
                print(f"Selected model: {model}")
                return model
        gpt_models = sorted([m for m in available if m.startswith("gpt-")], reverse=True)
        if gpt_models:
            print(f"Selected model: {gpt_models[0]}")
            return gpt_models[0]
    except Exception as e:
        print(f"Could not list models: {e}")
    fallback = "gpt-3.5-turbo"
    print(f"Fallback model: {fallback}")
    return fallback

BASE_MODEL = get_best_available_model()
print(f"Judge model: {JUDGE_MODEL}")
print("Setup complete.")


## Part 0: Offline Evaluation -- Your Unit Test Suite

Before we evaluate Chapter 6 and 7 outputs, this cell shows what offline evaluation looks like in practice.
We build a small golden dataset (5 questions with known correct answers), run a model against it,
score each response, then change the prompt and watch scores change.

This is exactly the regression detection loop described in the chapter:
change something, run the suite, see what broke.

In [ ]:
# === OFFLINE EVALUATION: Golden Dataset + Regression Detection ===

# Step 1: Build a small golden dataset (fixed test cases with known answers)
golden_dataset = [
    {
        "query": "What is the refund policy for digital products?",
        "reference_answer": "Digital products can be refunded within 14 days if not downloaded.",
        "category": "policy",
        "difficulty": "easy"
    },
    {
        "query": "Can I return a physical item after 30 days?",
        "reference_answer": "Physical items must be returned within 30 days with original packaging.",
        "category": "policy",
        "difficulty": "easy"
    },
    {
        "query": "What happens if my subscription renews and I want to cancel?",
        "reference_answer": "You can cancel within 48 hours of renewal for a full refund. After that, the subscription runs until the end of the billing period.",
        "category": "policy",
        "difficulty": "medium"
    },
    {
        "query": "I bought a gift card and the recipient lost it. Can I get a replacement?",
        "reference_answer": "Lost gift cards cannot be replaced unless you have the original receipt and card number.",
        "category": "edge_case",
        "difficulty": "hard"
    },
    {
        "query": "My order arrived damaged. Who pays for return shipping?",
        "reference_answer": "For damaged items, we provide a prepaid return label at no cost to you.",
        "category": "edge_case",
        "difficulty": "medium"
    }
]

print(f'Golden dataset: {len(golden_dataset)} test cases')
print(f'Categories: {set(g["category"] for g in golden_dataset)}')
print(f'Difficulties: {set(g["difficulty"] for g in golden_dataset)}')

Golden dataset: 5 test cases
Categories: {'edge_case', 'policy'}
Difficulties: {'easy', 'hard', 'medium'}

=== Prompt A: With policy context ===
  [5/5] What is the refund policy for digital products?
         The response matches the reference perfectly.
  [1/5] Can I return a physical item after 30 days?
         The response does not address the question and provides no relevant information.
  [3/5] What happens if my subscription renews and I want 
         The response correctly states that the subscription runs until the end of the billing period but omits the cancellation and refund details.
  [5/5] I bought a gift card and the recipient lost it. Ca
         The response matches the reference perfectly.
  [5/5] My order arrived damaged. Who pays for return ship
         The response matches the reference perfectly.

  Average score: 3.80/5

=== Prompt B: Without policy context ===
  [2/5] What is the refund policy for digital products?
         The response does not mention the 

In [ ]:
# Step 2: Define two prompts -- one good, one intentionally worse
PROMPT_A = """You are a customer support assistant. Answer the question using ONLY the provided policy.
If the answer is not in the policy, say "I don't have that information."

Policy: {reference}

Question: {query}"""

PROMPT_B = """Answer this customer question.

Question: {query}"""



In [ ]:
# Step 3: Run both prompts through the model and score
def run_offline_eval(prompt_template, golden_data, label):
    """Run one offline evaluation pass. Returns scores per example."""
    results = []
    for item in golden_data:
        # Build prompt
        if '{reference}' in prompt_template:
            prompt = prompt_template.format(query=item['query'], reference=item['reference_answer'])
        else:
            prompt = prompt_template.format(query=item['query'])

        # Get model response
        resp = client.chat.completions.create(
            model=BASE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0
        )
        model_answer = resp.choices[0].message.content.strip()

        # Score with LLM judge
        judge_prompt = f"""Score this response against the reference on a 1-5 scale.
1 = completely wrong, 5 = matches reference perfectly.

Question: {item['query']}
Reference: {item['reference_answer']}
Response: {model_answer}

Return ONLY a JSON object: {{"score": N, "reason": "one sentence"}}"""

        judge_resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": judge_prompt}],
            temperature=0.0
        )
        content = judge_resp.choices[0].message.content.strip()
        if '```json' in content:
            content = content.split('```json')[1].split('```')[0].strip()
        elif '```' in content:
            content = content.split('```')[1].split('```')[0].strip()
        scores = json.loads(content)
        scores['query'] = item['query'][:50]
        scores['category'] = item['category']
        scores['answer_preview'] = model_answer[:80]
        results.append(scores)

    return pd.DataFrame(results)


# Run Prompt A (with context)
print('\n=== Prompt A: With policy context ===')
results_a = run_offline_eval(PROMPT_A, golden_dataset, 'A')
for _, row in results_a.iterrows():
    print(f'  [{row["score"]}/5] {row["query"]}')
    print(f'         {row["reason"]}')

print(f'\n  Average score: {results_a["score"].mean():.2f}/5')

# Run Prompt B (no context -- should score worse)
print('\n=== Prompt B: Without policy context ===')
results_b = run_offline_eval(PROMPT_B, golden_dataset, 'B')
for _, row in results_b.iterrows():
    print(f'  [{row["score"]}/5] {row["query"]}')
    print(f'         {row["reason"]}')

print(f'\n  Average score: {results_b["score"].mean():.2f}/5')



In [ ]:
# Step 4: Regression report
print('\n' + '=' * 50)
print('REGRESSION REPORT: Prompt A vs Prompt B')
print('=' * 50)
delta = results_a['score'].mean() - results_b['score'].mean()
print(f'Prompt A average: {results_a["score"].mean():.2f}/5')
print(f'Prompt B average: {results_b["score"].mean():.2f}/5')
print(f'Delta:            {delta:+.2f}')
print()

# Find regressions (questions where B scored lower)
for i in range(len(golden_dataset)):
    diff = results_a.iloc[i]['score'] - results_b.iloc[i]['score']
    if diff > 0:
        print(f'  REGRESSION: "{golden_dataset[i]["query"][:45]}..."')
        print(f'    A={results_a.iloc[i]["score"]} -> B={results_b.iloc[i]["score"]} (dropped {diff} points)')
    elif diff < 0:
        print(f'  IMPROVEMENT: "{golden_dataset[i]["query"][:45]}..."')
        print(f'    A={results_a.iloc[i]["score"]} -> B={results_b.iloc[i]["score"]} (gained {abs(diff)} points)')

print()
if delta > 0:
    print(f'Prompt A wins by {delta:.2f} points. Removing context caused regressions.')
elif delta < 0:
    print(f'Prompt B wins by {abs(delta):.2f} points. Context may be confusing the model.')
else:
    print('No difference. The prompt change had no measurable effect.')

## Part 1: Loading Chapter 6 Datasets

Chapter 6 generated four datasets. We load them and check what we are working with.

In [12]:
# Load all four Chapter 6 datasets
ch6_path = Path('../../chapter_06/datasets')

hr_qa = pd.read_csv(ch6_path / 'hr_policy_qa_dataset.csv')
structured_qa = pd.read_csv(ch6_path / 'structured_qa_dataset.csv')
preference_pairs = pd.read_csv(ch6_path / 'preference_pairs_dataset.csv')
augmented_text = pd.read_csv(ch6_path / 'augmented_text_dataset.csv')

print('Chapter 6 datasets loaded:')
print(f'  HR QA pairs:        {len(hr_qa)} rows')
print(f'  Structured QA:      {len(structured_qa)} rows')
print(f'  Preference pairs:   {len(preference_pairs)} rows')
print(f'  Augmented text:     {len(augmented_text)} rows')
print()
print('HR QA sample:')
hr_qa.head(3)

Chapter 6 datasets loaded:
  HR QA pairs:        6 rows
  Structured QA:      12 rows
  Preference pairs:   4 rows
  Augmented text:     25 rows

HR QA sample:


,question,answer,source_policy
0,How many days of paid vacation do employees re...,Employees receive 15 days of paid vacation per...,POL-001 - Vacation Policy
1,How far in advance should employees request va...,Employees should request vacation time at leas...,POL-001 - Vacation Policy
2,How many days per week can employees work remo...,Employees can work remotely up to 3 days per w...,POL-002 - Remote Work Policy


## Part 2: Evaluating Synthetic QA Quality

The first question about any synthetic dataset: is it faithful to the source?
A generated answer that sounds plausible but adds details not in the source policy is a hallucination.
We use LLM-as-a-judge to score each QA pair on three dimensions.

In [13]:
def judge_qa_faithfulness(question, answer, source):
    """Score a QA pair on faithfulness, relevance, and accuracy using LLM-as-judge.
    Returns dict with scores 1-5 and justification.
    This implements the Siddall (2024) evaluation framework from the chapter."""

    prompt = f"""You are an evaluation judge. Score this QA pair on three dimensions.
Each score is 1-5 where 5 is best.

Source policy: {source}
Question: {question}
Answer: {answer}

Score these three dimensions:
1. Faithfulness: Does the answer ONLY use information from the source? (5 = fully grounded, 1 = hallucinated)
2. Relevance: Does the answer address the question? (5 = directly answers, 1 = off topic)
3. Accuracy: Is the answer factually correct given the source? (5 = fully correct, 1 = wrong)

Return ONLY valid JSON:
{{"faithfulness": N, "relevance": N, "accuracy": N, "justification": "one sentence"}}"""

    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )

    content = response.choices[0].message.content.strip()
    # Extract JSON
    if '```json' in content:
        content = content.split('```json')[1].split('```')[0].strip()
    elif '```' in content:
        content = content.split('```')[1].split('```')[0].strip()
    return json.loads(content)


# Evaluate all HR QA pairs
print('Evaluating HR QA pairs with LLM-as-judge...')
print()

results = []
for i, row in hr_qa.iterrows():
    scores = judge_qa_faithfulness(
        row['question'],
        row['answer'],
        row['source_policy']
    )
    scores['question'] = row['question'][:60]
    results.append(scores)
    print(f"  [{i+1}/{len(hr_qa)}] Faith={scores['faithfulness']} Rel={scores['relevance']} Acc={scores['accuracy']}")

eval_df = pd.DataFrame(results)

print()
print('=== Faithfulness Report ===')
print(f"Average faithfulness: {eval_df['faithfulness'].mean():.2f} / 5.00")
print(f"Average relevance:   {eval_df['relevance'].mean():.2f} / 5.00")
print(f"Average accuracy:    {eval_df['accuracy'].mean():.2f} / 5.00")
print(f"Pairs scoring below 4 on faithfulness: {(eval_df['faithfulness'] < 4).sum()}")
print()

# Show any problematic pairs
low_faith = eval_df[eval_df['faithfulness'] < 4]
if len(low_faith) > 0:
    print('Potentially hallucinated answers:')
    for _, row in low_faith.iterrows():
        print(f"  Q: {row['question']}")
        print(f"  Justification: {row['justification']}")
        print()
else:
    print('All QA pairs are faithful to the source. Good.')

Evaluating HR QA pairs with LLM-as-judge...

  [1/6] Faith=5 Rel=5 Acc=5
  [2/6] Faith=5 Rel=5 Acc=5
  [3/6] Faith=5 Rel=5 Acc=5
  [4/6] Faith=5 Rel=5 Acc=5
  [5/6] Faith=5 Rel=5 Acc=5
  [6/6] Faith=5 Rel=5 Acc=5

=== Faithfulness Report ===
Average faithfulness: 5.00 / 5.00
Average relevance:   5.00 / 5.00
Average accuracy:    5.00 / 5.00
Pairs scoring below 4 on faithfulness: 0

All QA pairs are faithful to the source. Good.


## Part 3: Evaluating QA Diversity

Faithful answers are useless if every question is a paraphrase of the same thing.
We measure diversity with pure Python, no API calls needed.
This is a rule-based metric, exactly the kind ROUGE and BLEU belong to.

In [14]:
def measure_diversity(questions):
    """Measure lexical diversity of a set of questions.
    Higher unique n-gram ratio = more diverse.
    Self-BLEU measures how similar questions are to each other (lower = better)."""

    all_unigrams = []
    all_bigrams = []

    for q in questions:
        tokens = q.lower().split()
        all_unigrams.extend(tokens)
        all_bigrams.extend(zip(tokens[:-1], tokens[1:]))

    unique_unigrams = len(set(all_unigrams))
    unique_bigrams = len(set(all_bigrams))
    total_unigrams = len(all_unigrams)
    total_bigrams = len(all_bigrams)

    # Type-token ratio
    ttr = unique_unigrams / total_unigrams if total_unigrams > 0 else 0
    bigram_ttr = unique_bigrams / total_bigrams if total_bigrams > 0 else 0

    # Question-start diversity: how many unique first 3 words?
    starters = [' '.join(q.lower().split()[:3]) for q in questions]
    starter_diversity = len(set(starters)) / len(starters) if starters else 0

    return {
        'total_questions': len(questions),
        'unique_unigrams': unique_unigrams,
        'unigram_ttr': round(ttr, 3),
        'unique_bigrams': unique_bigrams,
        'bigram_ttr': round(bigram_ttr, 3),
        'starter_diversity': round(starter_diversity, 3),
        'unique_starters': len(set(starters))
    }


# Measure diversity for both QA datasets
print('=== HR QA Diversity ===')
hr_div = measure_diversity(hr_qa['question'].tolist())
for k, v in hr_div.items():
    print(f'  {k}: {v}')

print()
print('=== Structured QA Diversity ===')
str_div = measure_diversity(structured_qa['question'].tolist())
for k, v in str_div.items():
    print(f'  {k}: {v}')

print()
# Interpretation
if hr_div['starter_diversity'] < 0.5:
    print('WARNING: Over half the HR questions start the same way.')
    print('The model may learn to pattern-match question openers instead of understanding the question.')
    print('Fix: regenerate with explicit diversity instructions in the prompt.')
else:
    print('Question starters are reasonably diverse. Good.')

if hr_div['unigram_ttr'] < 0.3:
    print('WARNING: Low vocabulary diversity. Questions reuse the same words heavily.')
else:
    print('Vocabulary diversity is acceptable.')

=== HR QA Diversity ===
  total_questions: 6
  unique_unigrams: 50
  unigram_ttr: 0.746
  unique_bigrams: 55
  bigram_ttr: 0.902
  starter_diversity: 0.667
  unique_starters: 4

=== Structured QA Diversity ===
  total_questions: 12
  unique_unigrams: 28
  unigram_ttr: 0.333
  unique_bigrams: 36
  bigram_ttr: 0.5
  starter_diversity: 0.417
  unique_starters: 5

Question starters are reasonably diverse. Good.
Vocabulary diversity is acceptable.


## Part 4: Evaluating Preference Pairs with Position Bias Check

Chapter 6 generated preferred vs non-preferred response pairs.
Now we test: can an LLM judge correctly identify which is better?
And does the judge show position bias, the tendency to favor whichever response comes first?
Siddall (2024) flagged this exact problem.

In [15]:
import random

def judge_preference(instruction, response_a, response_b, a_is_preferred):
    """Ask LLM judge to pick the better response.
    We randomize order to detect position bias.
    Returns dict with judge choice and whether it matches the label."""

    # Randomize presentation order to test for position bias
    show_preferred_first = random.random() < 0.5

    if show_preferred_first:
        first, second = response_a, response_b
        correct_choice = 'A'
    else:
        first, second = response_b, response_a
        correct_choice = 'B'

    prompt = f"""Which response is better for this instruction? Answer with just A or B.

Instruction: {instruction}

Response A: {first}

Response B: {second}

Better response (A or B):"""

    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=5
    )

    judge_pick = response.choices[0].message.content.strip().upper()
    # Extract just A or B
    if 'A' in judge_pick and 'B' not in judge_pick:
        judge_pick = 'A'
    elif 'B' in judge_pick and 'A' not in judge_pick:
        judge_pick = 'B'
    else:
        judge_pick = judge_pick[0] if judge_pick else '?'

    return {
        'correct': judge_pick == correct_choice,
        'judge_pick': judge_pick,
        'preferred_shown_first': show_preferred_first,
        'picked_first': judge_pick == 'A'
    }


print('Evaluating preference pairs with position bias check...')
print()

pref_results = []
for i, row in preference_pairs.iterrows():
    result = judge_preference(
        row['instruction'],
        row['preferred_response'],
        row['non_preferred_response'],
        a_is_preferred=True
    )
    pref_results.append(result)
    status = 'correct' if result['correct'] else 'WRONG'
    print(f"  [{i+1}/{len(preference_pairs)}] Judge picked: {result['judge_pick']} ({status})")

pref_df = pd.DataFrame(pref_results)

accuracy = pref_df['correct'].mean()
position_bias = pref_df['picked_first'].mean()

print()
print('=== Preference Evaluation Report ===')
print(f'Judge accuracy: {accuracy:.1%} ({pref_df["correct"].sum()}/{len(pref_df)} correct)')
print(f'Position bias:  {position_bias:.1%} picked Response A (first position)')
print(f'  (50% = no bias, >70% = strong position bias)')
print()

if accuracy < 0.7:
    print('WARNING: Judge accuracy below 70%. Either the preference pairs are too subtle')
    print('or the judge model is not strong enough. Try a stronger judge or clearer pairs.')
if position_bias > 0.7:
    print('WARNING: Strong position bias detected. The judge favors whichever response')
    print('appears first. This is exactly what Siddall (2024) warned about.')
    print('Fix: always randomize order and average across orderings.')

Evaluating preference pairs with position bias check...

  [1/4] Judge picked: A (correct)
  [2/4] Judge picked: B (correct)
  [3/4] Judge picked: B (correct)
  [4/4] Judge picked: B (correct)

=== Preference Evaluation Report ===
Judge accuracy: 100.0% (4/4 correct)
Position bias:  25.0% picked Response A (first position)
  (50% = no bias, >70% = strong position bias)



## Part 5: Loading and Inspecting Chapter 7 SFT Data

Chapter 7 fine-tuned a model using SFT data. Before we evaluate the model,
we check the training data itself. Bad training data produces bad models
regardless of the training technique.

In [16]:
# Load Chapter 7 SFT datasets
ch7_path = Path('../../chapter_07/datasets')

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            records.append(json.loads(line))
    return records

sft_train = load_jsonl(ch7_path / 'sft_train.jsonl')
sft_valid = load_jsonl(ch7_path / 'sft_valid.jsonl')

print(f'SFT training examples:   {len(sft_train)}')
print(f'SFT validation examples: {len(sft_valid)}')
print()

# Check behavioral consistency: do all examples use the same system prompt?
system_prompts = set()
for record in sft_train + sft_valid:
    for msg in record['messages']:
        if msg['role'] == 'system':
            system_prompts.add(msg['content'])

print(f'Unique system prompts: {len(system_prompts)}')
if len(system_prompts) == 1:
    print('Good: all examples share one behavioral contract.')
    print(f'System prompt: "{list(system_prompts)[0][:100]}..."')
else:
    print('WARNING: Multiple system prompts found. This can cause conflicting behavior.')
    for sp in system_prompts:
        print(f'  - "{sp[:80]}..."')

print()

# Check output structure consistency
print('=== Output Structure Check ===')
has_summary = 0
has_next_steps = 0
has_risks = 0

for record in sft_train:
    assistant_msg = [m for m in record['messages'] if m['role'] == 'assistant'][0]['content']
    if 'Summary' in assistant_msg or 'summary' in assistant_msg:
        has_summary += 1
    if 'Next steps' in assistant_msg or 'next steps' in assistant_msg:
        has_next_steps += 1
    if 'Risks' in assistant_msg or 'risks' in assistant_msg:
        has_risks += 1

print(f'Examples with Summary section:    {has_summary}/{len(sft_train)}')
print(f'Examples with Next Steps section: {has_next_steps}/{len(sft_train)}')
print(f'Examples with Risks section:      {has_risks}/{len(sft_train)}')
print()

consistency = min(has_summary, has_next_steps, has_risks) / len(sft_train)
if consistency > 0.8:
    print(f'Output structure consistency: {consistency:.0%}. The behavioral contract is clear.')
else:
    print(f'Output structure consistency: {consistency:.0%}. Some examples deviate from the contract.')
    print('This is exactly the kind of inconsistency that Chapter 7 warns about.')

SFT training examples:   10
SFT validation examples: 3

Unique system prompts: 1
Good: all examples share one behavioral contract.
System prompt: "You are a policy-aware support assistant. If context is insufficient, say 'Information not available..."

=== Output Structure Check ===
Examples with Summary section:    10/10
Examples with Next Steps section: 10/10
Examples with Risks section:      10/10

Output structure consistency: 100%. The behavioral contract is clear.


## Part 6: Evaluating Base Model vs the Behavioral Contract

We take the validation prompts from Chapter 7 and run them through the base model.
Then we score the outputs against the behavioral contract.
This gives us the baseline that fine-tuning is supposed to improve.

In [17]:
def evaluate_against_contract(messages, model_name):
    """Run a prompt through a model and score the output against the behavioral contract.
    The contract from Chapter 7 SFT data: Summary, Next Steps, Risks sections."""

    # Get model response
    system_msg = [m for m in messages if m['role'] == 'system']
    user_msg = [m for m in messages if m['role'] == 'user']

    response = client.chat.completions.create(
        model=model_name,
        messages=system_msg + user_msg,
        temperature=0.0
    )

    output = response.choices[0].message.content.strip()

    # Score against behavioral contract
    scores = {
        'has_summary': 1 if ('Summary' in output or 'summary' in output) else 0,
        'has_next_steps': 1 if ('Next steps' in output or 'next steps' in output or 'Next Steps' in output) else 0,
        'has_risks': 1 if ('Risks' in output or 'risks' in output or 'Risk' in output) else 0,
        'output_length': len(output),
        'output_preview': output[:200]
    }
    scores['contract_compliance'] = (scores['has_summary'] + scores['has_next_steps'] + scores['has_risks']) / 3

    return scores


print(f'Evaluating base model ({BASE_MODEL}) on Chapter 7 validation prompts...')
print()

base_results = []
for i, record in enumerate(sft_valid):
    scores = evaluate_against_contract(record['messages'], BASE_MODEL)
    base_results.append(scores)
    compliance = scores['contract_compliance']
    print(f'  [{i+1}/{len(sft_valid)}] Contract compliance: {compliance:.0%}')
    print(f'    Summary: {"yes" if scores["has_summary"] else "NO"}  '
          f'Next Steps: {"yes" if scores["has_next_steps"] else "NO"}  '
          f'Risks: {"yes" if scores["has_risks"] else "NO"}')
    print(f'    Preview: {scores["output_preview"][:100]}...')
    print()

base_df = pd.DataFrame(base_results)
avg_compliance = base_df['contract_compliance'].mean()

print('=== Base Model Evaluation ===')
print(f'Average contract compliance: {avg_compliance:.0%}')
print(f'  Summary present:    {base_df["has_summary"].mean():.0%}')
print(f'  Next Steps present: {base_df["has_next_steps"].mean():.0%}')
print(f'  Risks present:      {base_df["has_risks"].mean():.0%}')
print()
print('This is the baseline. If fine-tuning from Chapter 7 worked,')
print('the fine-tuned model should score higher on contract compliance.')

Evaluating base model (gpt-3.5-turbo-0125) on Chapter 7 validation prompts...

  [1/3] Contract compliance: 100%
    Summary: yes  Next Steps: yes  Risks: yes
    Preview: Summary: Enabling the new cache caused a spike in latency, which was resolved by disabling the cache...

  [2/3] Contract compliance: 0%
    Summary: NO  Next Steps: NO  Risks: NO
    Preview: Information not available. Are you looking to reset passwords for all users in a specific system or ...

  [3/3] Contract compliance: 100%
    Summary: yes  Next Steps: yes  Risks: yes
    Preview: Summary: The message queue depth reached 500K during peak traffic, but the consumer group was scaled...

=== Base Model Evaluation ===
Average contract compliance: 67%
  Summary present:    67%
  Next Steps present: 67%
  Risks present:      67%

This is the baseline. If fine-tuning from Chapter 7 worked,
the fine-tuned model should score higher on contract compliance.


## Part 7: LLM-as-Judge Scoring (Multi-Attribute)

Beyond structural compliance, we score the base model outputs using
Siddall's eight-attribute framework. This is the same framework
Clearwater uses in production. We pick four attributes most relevant
to the support assistant use case from Chapter 7.

In [18]:
def multi_attribute_judge(question, response, reference_answer):
    """Score a model response using multiple attributes from Siddall's framework.
    Uses a second LLM as judge with justification required (reduces bias)."""

    prompt = f"""You are an evaluation judge. Score this response on four attributes (1-5 each).
You MUST provide a one-sentence justification for each score.

Question: {question}
Reference answer: {reference_answer}
Model response: {response}

Attributes:
1. Accuracy: How closely does the response match the reference answer?
2. Relevance: Does the response directly address the question?
3. Coherence: Is the response logically structured and internally consistent?
4. Reasoning: How well does the model support its statements?

Return ONLY valid JSON:
{{"accuracy": N, "accuracy_reason": "...",
  "relevance": N, "relevance_reason": "...",
  "coherence": N, "coherence_reason": "...",
  "reasoning": N, "reasoning_reason": "...",
  "overall": N}}"""

    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )

    content = resp.choices[0].message.content.strip()
    if '```json' in content:
        content = content.split('```json')[1].split('```')[0].strip()
    elif '```' in content:
        content = content.split('```')[1].split('```')[0].strip()
    return json.loads(content)


print('Running multi-attribute evaluation on validation examples...')
print()

judge_results = []
for i, record in enumerate(sft_valid):
    user_msg = [m for m in record['messages'] if m['role'] == 'user'][0]['content']
    ref_answer = [m for m in record['messages'] if m['role'] == 'assistant'][0]['content']
    model_output = base_results[i]['output_preview']

    scores = multi_attribute_judge(user_msg, model_output, ref_answer)
    judge_results.append(scores)

    print(f'  [{i+1}/{len(sft_valid)}] Acc={scores["accuracy"]} Rel={scores["relevance"]} '
          f'Coh={scores["coherence"]} Reas={scores["reasoning"]} Overall={scores["overall"]}')
    print(f'    Accuracy reason: {scores["accuracy_reason"]}')

judge_df = pd.DataFrame(judge_results)

print()
print('=== Multi-Attribute Evaluation Report ===')
for attr in ['accuracy', 'relevance', 'coherence', 'reasoning', 'overall']:
    print(f'  {attr:12s}: {judge_df[attr].mean():.2f} / 5.00')

Running multi-attribute evaluation on validation examples...

  [1/3] Acc=5 Rel=5 Coh=4 Reas=4 Overall=4
    Accuracy reason: The response accurately summarizes the situation regarding latency and cache performance as described in the question.
  [2/3] Acc=4 Rel=4 Coh=5 Reas=5 Overall=4
    Accuracy reason: The response accurately identifies the need for clarification and security considerations but lacks specific details about the bulk password reset process.
  [3/3] Acc=4 Rel=4 Coh=4 Reas=3 Overall=4
    Accuracy reason: The model response accurately summarizes the key points but omits specific details about the auto-scaling and next steps.

=== Multi-Attribute Evaluation Report ===
  accuracy    : 4.33 / 5.00
  relevance   : 4.33 / 5.00
  coherence   : 4.33 / 5.00
  reasoning   : 4.00 / 5.00
  overall     : 4.00 / 5.00


## Part 8: The Complete Evaluation Report

Pull everything together into the 2x2 evaluation matrix from the chapter.
This is what a real evaluation report looks like.

In [19]:
print('=' * 60)
print('CHAPTER 9 EVALUATION REPORT')
print('=' * 60)
print()
print('--- Chapter 6: Synthetic Data Quality ---')
print()
print('QA Faithfulness (LLM-as-judge):')
print(f'  Average faithfulness: {eval_df["faithfulness"].mean():.2f}/5')
print(f'  Average relevance:   {eval_df["relevance"].mean():.2f}/5')
print(f'  Average accuracy:    {eval_df["accuracy"].mean():.2f}/5')
print(f'  Hallucinated pairs:  {(eval_df["faithfulness"] < 4).sum()}/{len(eval_df)}')
print()
print('QA Diversity (rule-based):')
print(f'  Unigram TTR:         {hr_div["unigram_ttr"]}')
print(f'  Starter diversity:   {hr_div["starter_diversity"]}')
print()
print('Preference Pairs (LLM-as-judge with position bias check):')
print(f'  Judge accuracy:      {pref_df["correct"].mean():.0%}')
print(f'  Position bias:       {pref_df["picked_first"].mean():.0%} picked first')
print()
print('--- Chapter 7: SFT Behavioral Contract ---')
print()
print('Training Data Consistency:')
print(f'  Unique system prompts:  {len(system_prompts)}')
print(f'  Structure consistency:  {consistency:.0%}')
print()
print(f'Base Model Contract Compliance ({BASE_MODEL}):')
print(f'  Average compliance:     {avg_compliance:.0%}')
print(f'  Summary present:        {base_df["has_summary"].mean():.0%}')
print(f'  Next Steps present:     {base_df["has_next_steps"].mean():.0%}')
print(f'  Risks present:          {base_df["has_risks"].mean():.0%}')
print()
print('Multi-Attribute Judge Scores (Siddall framework):')
for attr in ['accuracy', 'relevance', 'coherence', 'reasoning', 'overall']:
    print(f'  {attr:12s}:          {judge_df[attr].mean():.2f}/5')
print()
print('=' * 60)
print('VERDICT')
print('=' * 60)
overall = judge_df['overall'].mean()
if overall >= 4.0 and avg_compliance >= 0.8:
    print('Base model already meets the behavioral contract.')
    print('Fine-tuning may not be necessary for this task.')
elif overall >= 3.0:
    print('Base model partially meets the contract.')
    print('Fine-tuning should improve consistency on the gaps.')
else:
    print('Base model does not meet the behavioral contract.')
    print('Fine-tuning from Chapter 7 is justified.')
print()
print('Next: run the same evaluation on the fine-tuned model from Chapter 7')
print('and compare the numbers. The delta is your fine-tuning ROI.')

CHAPTER 9 EVALUATION REPORT

--- Chapter 6: Synthetic Data Quality ---

QA Faithfulness (LLM-as-judge):
  Average faithfulness: 5.00/5
  Average relevance:   5.00/5
  Average accuracy:    5.00/5
  Hallucinated pairs:  0/6

QA Diversity (rule-based):
  Unigram TTR:         0.746
  Starter diversity:   0.667

Preference Pairs (LLM-as-judge with position bias check):
  Judge accuracy:      100%
  Position bias:       25% picked first

--- Chapter 7: SFT Behavioral Contract ---

Training Data Consistency:
  Unique system prompts:  1
  Structure consistency:  100%

Base Model Contract Compliance (gpt-3.5-turbo-0125):
  Average compliance:     67%
  Summary present:        67%
  Next Steps present:     67%
  Risks present:          67%

Multi-Attribute Judge Scores (Siddall framework):
  accuracy    :          4.33/5
  relevance   :          4.33/5
  coherence   :          4.33/5
  reasoning   :          4.00/5
  overall     :          4.00/5

VERDICT
Base model partially meets the contract.